# ML-10 — Content Action Playbook
Hicham Halfi · 2026-10-20

## 1. Ranked actions and reason codes
This notebook uses the prior-window model assessed in ML-09. It generates out-of-fold scores: every row is scored by a forest trained without its client. These are retrospective review scores, not calibrated probabilities or predictions of refresh benefit. The queue is local and ignored by Git; only aggregate metrics and figures are published.

The model ranks review candidates. A separate, explicit heuristic maps current observable content state to a proposed review action. Reason codes describe those observations, not causal explanations of the forest. Age/freshness are used for editorial context only, never as historical model features.


In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/raw/content_refresh_anonymized.csv').exists())
OUT=ROOT/'work/outputs'; OUT.mkdir(exist_ok=True)
FIG=ROOT/'work/figures'; FIG.mkdir(exist_ok=True)
audit=json.loads((OUT/'validation_metrics.json').read_text())
path=ROOT/'data/raw/content_refresh_anonymized.csv'
assert hashlib.sha256(path.read_bytes()).hexdigest()==audit['dataset_sha256']
df=pd.read_csv(path); df=df.loc[df.impressions_90d.gt(0)].reset_index(drop=True)
y=df.trend_direction.str.lower().eq('down').astype(int)
features=['impressions_prev_30d','clicks_prev_30d','sessions_prev_30d']
scores=np.full(len(df),np.nan)
for train,test in GroupKFold(5).split(df,y,df.client_id):
    assert not set(df.iloc[train].client_id)&set(df.iloc[test].client_id)
    model=RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42,n_jobs=1)
    model.fit(df.iloc[train][features],y.iloc[train])
    scores[test]=model.predict_proba(df.iloc[test][features])[:,1]
assert np.isfinite(scores).all()

def propose(row):
    if row.impressions_prev_30d<100:
        return 'Low prior evidence','COLLECT_MORE_EVIDENCE','Check tracking and demand before editing',.5
    if row.avg_position==0:
        return 'Missing position','POSITION_MISSING','Verify measurement before any content action',.5
    if row.content_age_days>=180 and row.days_since_last_update>=90:
        return 'Mature and stale','AGE_180_PLUS;UPDATE_90_PLUS','Review outdated facts and intent; propose targeted refresh',2.
    if row.avg_position<=10 and row.ctr<.5:
        return 'Visible, weak click capture','PAGE_ONE;CTR_BELOW_0_5_PCT','Review title and snippet against intent',1.
    if 10<row.avg_position<=20:
        return 'Near page one','POSITION_11_TO_20','Review relevance gaps and internal links',1.5
    return 'Other visible content','GENERAL_REVIEW','Inspect trend and content quality before deciding',1.

mapped=pd.DataFrame([propose(r) for r in df.itertuples()],columns=['archetype','reason_codes','proposed_action','review_hours_assumption'])
queue=pd.concat([df[['content_id','client_id']],mapped],axis=1)
queue['oof_review_score']=scores
queue=queue.sort_values('oof_review_score',ascending=False,kind='stable').reset_index(drop=True)
queue.insert(0,'rank',np.arange(1,len(queue)+1))
queue['human_approval_required']=True
queue['status']='proposed_not_executed'
queue.to_csv(OUT/'ranked_action_queue.csv',index=False)
top_indices=np.argsort(-scores,kind='stable')[:20]
p20=float(y.iloc[top_indices].mean())
print('OOF queue rows:',len(queue),'P@20:',p20,'prevalence:',round(y.mean(),4))
print('Top 20 archetype counts:',queue.head(20).archetype.value_counts().to_dict())
print('Top 20 assumed review hours:',float(queue.head(20).review_hours_assumption.sum()))


OOF queue rows: 30000 P@20: 0.75 prevalence: 0.5421
Top 20 archetype counts: {'Visible, weak click capture': 8, 'Mature and stale': 6, 'Other visible content': 4, 'Near page one': 2}
Top 20 assumed review hours: 27.0


## 2. Intended use, decay insight, and limits
Intended user: a content editor planning a small review queue, with an analyst checking measurement quality. Review ten candidates per week initially. The source label is a contemporaneous >20% impression decline; it is not bad-content truth and does not tell us which edit will help. No warehouse-scale or future generalization claim is made.

The source research paper's freshness comparison (page 9) is an observational comparison of selected refreshed and unrefreshed pages. It motivates inspecting mature stale assets, but cannot justify promising a multiplier after an edit. The notebook's archetypes are transparent heuristics, not a reproduction of the paper's k-means clusters. Thresholds are editorial assumptions, not optimized cutoffs. The following table checks whether our sample even supports a simple stale/decline association; it does not estimate a treatment effect.


In [2]:
stale=df.days_since_last_update.ge(90)
summary=pd.DataFrame({'stale_90_plus':stale,'decline':y}).groupby('stale_90_plus').decline.agg(['size','mean'])
print('Observed snapshot decline by recency; unadjusted, not causal:')
print(summary.to_string())


Observed snapshot decline by recency; unadjusted, not causal:
                size      mean
stale_90_plus                 
False          20655  0.512031
True            9345  0.608454


## 3. Human review, cost/value, and no-go list
Before accepting any candidate, the editor checks tracking, seasonality, business relevance, accuracy and intent. A subject expert approves factual changes. An owner decides whether to publish. Log the decision, reason, effort, and follow-up; a high score alone must never trigger rewriting, deletion, consolidation, publication, redirects, outreach, or changes to analytics.

No-go: no raw client identification, no generated factual claims without verification, no automated publication or destructive action, no guaranteed traffic/revenue uplift, no inference about a particular search algorithm. Low prior volume and missing position require measurement review first. The queue does not contain article text, so it cannot diagnose factual gaps by itself.

Cost/value scenario (assumptions, not measured outcomes): reserve up to ten editor-hours per week. At an assumed $30/hour, that is a $300 weekly review budget. The per-row hours are planning estimates only. Choose the highest-ranked mix that fits the budget after human triage; stop if evidence is insufficient. Value remains unknown until a prospective matched or randomized evaluation measures incremental clicks and business outcomes. P@20 measures retrieval of decline labels, not ROI.

## 4. Monitoring and retrain triggers
For a prospective pilot, first establish dated feature/label windows. Weekly: validate schema, missingness and position-zero frequency; pause if required columns disappear, >5% retained values are missing, or >10% of numeric values violate documented domains. Compare prior-impression median and zero-volume share with the development reference; >50% median shift or >10 percentage-point zero-share change triggers investigation, not automatic retraining.

After outcome windows mature, monitor prevalence and P@20 by client alongside the baseline. Investigate a decline of >=10 percentage points across three completed batches, or baseline parity. Refit only with owner approval, a logged version, new grouped/time evaluation, and a rollback candidate. Do not retrain on editorial decisions as if they were verified labels. Thresholds are proposed operating rules and have not been tested in production.

## 5. Exports for the paper


In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
counts=queue.head(20).archetype.value_counts()
fig,ax=plt.subplots(figsize=(9,4.8))
ax.barh(counts.index,counts.values,color='#287d8e'); ax.invert_yaxis()
ax.set(xlabel='Candidates in top 20',title='Human review proposals from out-of-client scores')
fig.tight_layout(); fig.savefig(FIG/'action_mix.png',dpi=180); plt.close(fig)
payload={'rows':len(queue),'oof_precision_at_20':p20,'base_rate':float(y.mean()),
 'top20_action_counts':{str(k):int(v) for k,v in counts.items()},
 'top20_assumed_hours':float(queue.head(20).review_hours_assumption.sum()),
 'thresholds_are_assumptions':True,'queue_path':'work/outputs/ranked_action_queue.csv',
 'prediction_scope':'retrospective out-of-client; no causal or future-performance claim'}
(OUT/'playbook_metrics.json').write_text(json.dumps(payload,indent=2))
assert queue.human_approval_required.all() and queue.status.eq('proposed_not_executed').all()
print('Saved local queue, aggregate metrics, and figure. No action was executed.')


Saved local queue, aggregate metrics, and figure. No action was executed.


## Self-check
- Ranked queue uses client-separated out-of-fold scoring and explicit reason codes.
- Archetype mapping, freshness caveat, intended use, limits, cost assumptions, and no-go rules included.
- Queue export is gitignored; aggregate JSON and chart are reproducible and publishable.
- Run all cells top to bottom before committing. This notebook proposes editorial decisions; no real-world content action has occurred.
